In [1]:
import jax
import jax.numpy as jnp
from flax import nnx

In [2]:
class TokenEmbedding(nnx.Module):
    def __init__(self, vocabulary_size: int, embedding_size: int, rngs: nnx.Rngs) -> None:
        self.embedding = nnx.Embed(num_embeddings=vocabulary_size, features=embedding_size, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        return self.embedding(x)

In [3]:
class PositionEmbedding(nnx.Module):
    def __init__(self, max_sequence_length: int, embedding_size: int, rngs: nnx.Rngs) -> None:
        self.embedding = nnx.Embed(num_embeddings=max_sequence_length, features=embedding_size, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        positions = jnp.arange(x.shape[1]).reshape(1, -1)
        return self.embedding(positions)

In [4]:
def create_attention_mask(size: int) -> jax.Array:
    return jnp.tril(jnp.ones((size, size)))

In [5]:
class TransformerBlock(nnx.Module):
    def __init__(self, num_heads: int, input_size: int, qkv_features: int, rngs: nnx.Rngs) -> None:
        self.attention = nnx.MultiHeadAttention(
            num_heads=num_heads,
            in_features=input_size,
            qkv_features=qkv_features,
            out_features=input_size,
            decode=False,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        tokens_len = x.shape[1]
        mask = create_attention_mask(tokens_len)
        return x + self.attention(x, mask=mask)

In [6]:
class Transformer(nnx.Module):
    def __init__(
        self,
        vocabulary_size: int,
        max_sequence_length: int,
        num_transformer_blocks: int,
        num_heads: int,
        embedding_size: int,
        rngs: nnx.Rngs,
    ) -> None:
        self.token_embedding = TokenEmbedding(vocabulary_size=vocabulary_size, embedding_size=embedding_size, rngs=rngs)
        self.position_embedding = PositionEmbedding(
            max_sequence_length=max_sequence_length, embedding_size=embedding_size, rngs=rngs
        )
        self.blocks = nnx.Sequential(
            *[
                TransformerBlock(num_heads=num_heads, input_size=embedding_size, qkv_features=embedding_size, rngs=rngs)
                for _ in range(num_transformer_blocks)
            ]
        )
        self.linear = nnx.Linear(embedding_size, vocabulary_size, use_bias=False, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        embedded = self.token_embedding(x) + self.position_embedding(x)
        trans = self.blocks(embedded)
        return self.linear(trans)